# LipPY Mel Cache Pre-compute (Whisper-large-v3, 128-bin, FP16)

Audio v2 (large-v3) eğitimini hızlandırmak için tüm wav dosyalarından log-mel spectrogram'ları **bir kez** hesaplar ve diske yazar. Audio v2 dataset class'ı daha sonra `feature_extractor(wav)` çağırmak yerine `torch.load(mel_path)` ile direkt cache'ten okur.

## Niye large-v3?

Medium ile ep5 ~%32-33 PER tahmin edildi (hedef %18-22'in altında). Sample kalitesi AV-Fusion için yetersiz. Large-v3 ile beklenen PER %22-26, hedef bandında.

## Sebep (cache)

Audio v2 ep1 cache'siz 11h+ sürüyor. Her sample CPU'da ~75-110ms (resample + Whisper FE + augmentation). Cache ile dataloader 15-25x hızlanır, large-v3 epoch ~2.5-3h iner.

## Strateji

- **CPU instance** kullan (GPU quota harcamasin, paralel olarak GPU notebook ile çalışabilir)
- **Multiprocessing** (Kaggle CPU = 4 core, 3 worker)
- **Idempotent**: var olan dosyalar atlanır, kesilen run devam ettirilebilir
- **FP16 disk format**: ~9 GB → ~4.5 GB tasarruf
- **Tek variant** (speed=1.0): training sırasında mel-domain time stretch ile speed perturb yapilabilir

## Tahmini süre

- 77K wav × ~120ms / 3 workers ≈ **50-70 dakika** CPU instance'da
- Upload: ~45-90 dakika (4.5 GB, Kaggle dataset)
- **Toplam: ~2-2.5 saat tek sefer**

Medium cache'inden ~%50 yavaş (128 mel vs 80 mel, daha buyuk dosya).

## Inputs (Kaggle notebook'a ekle)

- `brokyewing/lippy-audio-segments` (wav dosyalari)

## Output

- `brokyewing/lippy-mel-cache-whisper-large-v3` (Kaggle dataset, ~4.5 GB)

In [ ]:
!pip install -q transformers torchaudio soundfile

import os, glob, json, subprocess, time
from multiprocessing import Pool, cpu_count
import torch
import torchaudio
from transformers import WhisperFeatureExtractor

# ============ CONFIG ============
# LARGE-V3: 128 mel bins (medium 80'di, ayri cache gerekti)
HF_MODEL = 'openai/whisper-large-v3'
SAMPLE_RATE = 16000

# Cache dosyalari upload dir'in icinde olsun, kopyalama derdi yok
UPLOAD_DIR = '/kaggle/working/upload_mel_cache'
MEL_CACHE_DIR = f'{UPLOAD_DIR}/mel_cache'
DATASET_SLUG = 'lippy-mel-cache-whisper-large-v3'   # medium'dan ayri slug

NUM_WORKERS = max(1, cpu_count() - 1)  # 1 core'u OS icin birak
PROGRESS_EVERY = 5000                    # her N sample'da progress yaz

os.makedirs(MEL_CACHE_DIR, exist_ok=True)
print(f'Model:        {HF_MODEL} (mel bins=128)')
print(f'CPU cores:    {cpu_count()} (workers={NUM_WORKERS})')
print(f'Cache dir:    {MEL_CACHE_DIR}')
print(f'Upload slug:  {DATASET_SLUG}')

# Kaggle credentials (upload icin gerekli)
KAGGLE_USER = os.environ.get('KAGGLE_USERNAME', 'UNKNOWN')
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    KAGGLE_USER = _us.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USER
    os.environ['KAGGLE_KEY'] = _us.get_secret('KAGGLE_KEY')
    print(f'Kaggle user:  {KAGGLE_USER}')
except Exception as e:
    print(f'UYARI: Kaggle secrets okunamadi: {e}')
    print('Upload step calismaz, manuel olarak indirip yukleyebilirsin')

DATASET_ID = f'{KAGGLE_USER}/{DATASET_SLUG}'
print(f'Upload target: {DATASET_ID}')

In [ ]:
# Audio dir auto-discovery (audio v2 ile ayni pattern)
AUDIO_CANDS = (
    glob.glob('/kaggle/input/**/lippy-audio-segments/**/audios', recursive=True)
    + glob.glob('/kaggle/input/**/audios', recursive=True)
)
AUDIO_DIR = AUDIO_CANDS[0] if AUDIO_CANDS else '/kaggle/input/lippy-audio-segments/audios'
print(f'AUDIO_DIR: {AUDIO_DIR}')

all_wavs = sorted(glob.glob(f'{AUDIO_DIR}/*.wav'))
print(f'Bulunan wav dosyasi: {len(all_wavs):,}')

# Resume kontrolu: var olan cache dosyalari
existing = set(
    os.path.basename(p)[:-3]  # .pt uzantisi cikar
    for p in glob.glob(f'{MEL_CACHE_DIR}/*.pt')
)
print(f'Mevcut cache: {len(existing):,} dosya')

# Sadece eksikleri isle
todo = [w for w in all_wavs if os.path.basename(w)[:-4] not in existing]
print(f'Islenecek:    {len(todo):,} dosya')
print(f'Tahmini sure: ~{len(todo) * 0.08 / NUM_WORKERS / 60:.1f} dakika (multiprocessing)')

if len(todo) == 0:
    print('\nTum cache hazir, dogrudan upload step\'ine gec.')

In [ ]:
# Worker initializer: her process kendi feature_extractor'unu yukler
# (HF FeatureExtractor pickle'lanamaz, her worker ayri instance kullanmali)
_fe = None

def init_worker():
    global _fe
    _fe = WhisperFeatureExtractor.from_pretrained(HF_MODEL)


def process_one(wav_path):
    """Tek wav dosyasini mel'e cevirip diske yazar. FP16 format."""
    sid = os.path.basename(wav_path)[:-4]
    out_path = f'{MEL_CACHE_DIR}/{sid}.pt'

    # Idempotent: var olan atla
    if os.path.exists(out_path):
        return ('skip', sid)

    try:
        # Load + resample + mono
        wav, sr = torchaudio.load(wav_path)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        if wav.shape[0] > 1:
            wav = wav.mean(0, keepdim=True)
        wav_np = wav.squeeze(0).float().numpy()

        # Whisper feature extraction (DYNAMIC length, padding=False)
        mel = _fe(
            wav_np,
            sampling_rate=SAMPLE_RATE,
            return_tensors='pt',
            padding=False,
            return_attention_mask=False,
        ).input_features
        mel = mel.squeeze(0).half()  # (80, T_mel), FP16 ile %50 disk tasarrufu

        # Atomic write: temp dosyaya yaz, sonra rename (interrupt safe)
        tmp_path = out_path + '.tmp'
        torch.save(mel, tmp_path)
        os.rename(tmp_path, out_path)
        return ('ok', sid)
    except Exception as e:
        return (f'err:{type(e).__name__}', sid)


# ============ Pre-compute main loop ============
if len(todo) > 0:
    t0 = time.time()
    print(f'Pre-compute basliyor: {len(todo):,} dosya, {NUM_WORKERS} worker')
    print()

    ok_count = skip_count = err_count = 0
    err_samples = []

    with Pool(NUM_WORKERS, initializer=init_worker) as pool:
        for i, (status, sid) in enumerate(pool.imap_unordered(process_one, todo, chunksize=20), 1):
            if status == 'ok':
                ok_count += 1
            elif status == 'skip':
                skip_count += 1
            else:
                err_count += 1
                if len(err_samples) < 10:
                    err_samples.append(f'{sid}: {status}')

            if i % PROGRESS_EVERY == 0:
                elapsed = time.time() - t0
                rate = i / elapsed
                eta_min = (len(todo) - i) / rate / 60
                print(f'  {i:>6,}/{len(todo):,} | {rate:.1f}/s | ETA: {eta_min:.1f} dk | OK={ok_count} Err={err_count}')

    elapsed = time.time() - t0
    print()
    print(f'TAMAMLANDI -- {elapsed/60:.1f} dakika')
    print(f'  OK:   {ok_count:,}')
    print(f'  Skip: {skip_count:,}  (zaten cache\'te)')
    print(f'  Err:  {err_count:,}')
    if err_samples:
        print(f'  Hata ornekleri:')
        for s in err_samples:
            print(f'    {s}')
else:
    print('Pre-compute atlandi (todo bos).')

# Final durum
final_count = len(glob.glob(f'{MEL_CACHE_DIR}/*.pt'))
print(f'\nToplam cache dosyasi: {final_count:,}')

In [ ]:
# Disk kullanimi kontrol
import shutil
total, used, free = shutil.disk_usage(MEL_CACHE_DIR)
cache_size_bytes = sum(
    os.path.getsize(os.path.join(MEL_CACHE_DIR, f))
    for f in os.listdir(MEL_CACHE_DIR)
    if f.endswith('.pt')
)
print(f'Cache disk kullanimi: {cache_size_bytes / 1e9:.2f} GB')
print(f'Working disk free:    {free / 1e9:.2f} GB')

# Sample mel inceleme (sanity check)
sample_files = glob.glob(f'{MEL_CACHE_DIR}/*.pt')[:3]
print(f'\nSample cache dosyalari:')
for f in sample_files:
    m = torch.load(f, weights_only=True)
    print(f'  {os.path.basename(f)}: shape={tuple(m.shape)}, dtype={m.dtype}, '
          f'min={m.float().min():.2f}, max={m.float().max():.2f}')

## Kaggle dataset olarak yukle

Bu cell sadece **tum dosyalar hazir** olduktan sonra calistirilmali (yukarida `Err=0` ve sayi 77K civari olmali).

**ILK kez**: `kaggle datasets create` -- dataset yoksa sifirdan yarat
**Sonraki update'ler**: `kaggle datasets version` -- yeni versiyon ekle

In [ ]:
# Dataset metadata
with open(f'{UPLOAD_DIR}/dataset-metadata.json', 'w') as f:
    json.dump({
        'title': 'LipPY Mel Cache (Whisper-large-v3 128-bin FP16)',
        'id': DATASET_ID,
        'licenses': [{'name': 'CC0-1.0'}],
    }, f)
print(f'Metadata yazildi: {UPLOAD_DIR}/dataset-metadata.json')

# Once create dene, exists ise version
print(f'\nUpload: {DATASET_ID} ...')
print(f'  (~4.5 GB, 45-90 dakika surebilir)')

r = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', UPLOAD_DIR, '--dir-mode', 'zip'],
    capture_output=True, text=True, timeout=7200)

print(f'CREATE rc={r.returncode}')
if r.stdout: print(f'  stdout: {r.stdout[-400:]}')
if r.stderr: print(f'  stderr: {r.stderr[-400:]}')

# Eger zaten varsa version dene (heuristic yerine her failure'da dene)
if r.returncode != 0:
    print('\nCREATE basarisiz, VERSION deneniyor (dataset zaten var ihtimali)...')
    vr = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', UPLOAD_DIR,
         '-m', 'mel cache pre-compute large-v3', '--dir-mode', 'zip'],
        capture_output=True, text=True, timeout=7200)
    print(f'VERSION rc={vr.returncode}')
    if vr.stdout: print(f'  stdout: {vr.stdout[-400:]}')
    if vr.stderr: print(f'  stderr: {vr.stderr[-400:]}')

print(f'\nTAMAMLANDI. Audio v2 notebook\'una input olarak ekle: {DATASET_ID}')